# GPT-Style Decoder-Only Transformer (PyTorch)

This notebook contains a pure PyTorch implementation of a GPT-style decoder-only Transformer for MIDI token sequences.

It includes:
- Model definition only (no dataset, preprocessing, or training code)
- Strict causal masking in self-attention
- A minimal forward pass test

In [ ]:
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class GPTConfig:
    vocab_size: int = 2048
    max_seq_len: int = 512
    d_model: int = 512
    n_layers: int = 6
    n_heads: int = 8
    d_ff: int = 2048
    dropout: float = 0.1


class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        if config.d_model % config.n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads.")

        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads
        self.scale = self.head_dim ** -0.5

        self.qkv_proj = nn.Linear(config.d_model, 3 * config.d_model)
        self.out_proj = nn.Linear(config.d_model, config.d_model)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bsz, seq_len, d_model = x.shape

        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(bsz, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(bsz, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(bsz, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        attn_scores = (q @ k.transpose(-2, -1)) * self.scale

        # Strict causal mask: token t only attends to <= t.
        causal_mask = torch.tril(
            torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool)
        )
        attn_scores = attn_scores.masked_fill(~causal_mask, float("-inf"))

        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        out = attn_weights @ v
        out = out.transpose(1, 2).contiguous().view(bsz, seq_len, d_model)
        out = self.resid_dropout(self.out_proj(out))
        return out


class TransformerBlock(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        self.ln1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.d_model)
        self.mlp = nn.Sequential(
            nn.Linear(config.d_model, config.d_ff),
            nn.GELU(),
            nn.Linear(config.d_ff, config.d_model),
            nn.Dropout(config.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class GPTMusicModel(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        self.config = config

        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_embedding = nn.Embedding(config.max_seq_len, config.d_model)
        self.dropout = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList(
            [TransformerBlock(config) for _ in range(config.n_layers)]
        )
        self.ln_final = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        bsz, seq_len = input_ids.shape
        if seq_len > self.config.max_seq_len:
            raise ValueError(
                f"Sequence length {seq_len} exceeds max_seq_len={self.config.max_seq_len}."
            )

        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        x = self.token_embedding(input_ids) + self.pos_embedding(positions)
        x = self.dropout(x)

        for block in self.blocks:
            x = block(x)

        x = self.ln_final(x)
        logits = self.lm_head(x)
        return logits

In [ ]:
# Minimal forward-pass test
config = GPTConfig(
    vocab_size=4096,
    max_seq_len=512,
    d_model=512,
    n_layers=6,
    n_heads=8,
    d_ff=2048,
    dropout=0.1,
)

model = GPTMusicModel(config)

batch_size = 2
seq_len = 128
input_ids = torch.randint(0, config.vocab_size, (batch_size, seq_len), dtype=torch.long)

with torch.no_grad():
    logits = model(input_ids)

print("Input shape:", input_ids.shape)
print("Logits shape:", logits.shape)  # Expected: [batch_size, seq_len, vocab_size]

## Minimal Training Cells

These cells implement next-token prediction training for preloaded token sequences.

Assumptions:
- `token_sequences` is already available in memory
- each entry is a 1D tensor/list of integer token IDs
- no dataset loading or preprocessing is done here

In [ ]:
import math


device = "cuda" if torch.cuda.is_available() else "cpu"
model = GPTMusicModel(config).to(device)

# Training hyperparameters
batch_size = 8
train_seq_len = 128
max_steps = 200
learning_rate = 3e-4
weight_decay = 0.01
warmup_steps = 20
grad_clip = 1.0
use_amp = (device == "cuda")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    betas=(0.9, 0.95),
    weight_decay=weight_decay,
)


def lr_lambda(step: int) -> float:
    if step < warmup_steps:
        return float(step + 1) / float(max(1, warmup_steps))
    progress = (step - warmup_steps) / float(max(1, max_steps - warmup_steps))
    progress = min(max(progress, 0.0), 1.0)
    return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
scaler = torch.amp.GradScaler(enabled=use_amp)


# Helper: random next-token training batch from in-memory sequences
# Expects token_sequences to be a list of integer sequences.
def get_batch(token_sequences, batch_size: int, seq_len: int, device: str):
    valid = [seq for seq in token_sequences if len(seq) >= seq_len + 1]
    if not valid:
        raise ValueError("No sequences long enough for the requested seq_len.")

    x_list = []
    y_list = []
    for _ in range(batch_size):
        seq = valid[torch.randint(0, len(valid), (1,)).item()]
        seq = torch.as_tensor(seq, dtype=torch.long)
        start = torch.randint(0, len(seq) - seq_len, (1,)).item()
        chunk = seq[start : start + seq_len + 1]
        x_list.append(chunk[:-1])
        y_list.append(chunk[1:])

    x = torch.stack(x_list).to(device)
    y = torch.stack(y_list).to(device)
    return x, y

In [ ]:
# Example token_sequences for a runnable demo.
# Replace this with your real preloaded MIDI token sequences.
if "token_sequences" not in globals():
    token_sequences = [
        torch.randint(0, config.vocab_size, (1024,), dtype=torch.long)
        for _ in range(64)
    ]

model.train()
for step in range(max_steps):
    x, y = get_batch(token_sequences, batch_size, train_seq_len, device)

    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, config.vocab_size), y.view(-1))

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

    if step % 20 == 0 or step == max_steps - 1:
        current_lr = scheduler.get_last_lr()[0]
        print(f"step={step:04d} loss={loss.item():.4f} lr={current_lr:.6f}")